In [ ]:
"""
Event-study Difference-in-Differences (DiD) for fire impacts on recreation
=======================================================================
    1) Pre-trend diagnostic summary (12-month pre-window)
    2) Logs the unique site IDs used in each DiD specification
    3) Logs panel metrics (treated/control sites, obs, etc.)

Required inputs (two CSVs):
1) predictions CSV (panel outcome + treatment timing):
   Must include (case-insensitive):
     - siteid
     - year, month
     - rel_month                (integer months relative to ignition; negative = pre)
     - treatment_type           (contains wildfire/rx and treated/control)
     - y_pred                   (default outcome; change OUTCOME_COL to use another)

2) fire/feature matrix CSV (time-varying controls + site covariates):
   Must include (case-insensitive):
     - siteid
     - year, month
   Optional (used if present):
     - temperature_mean         (or other weather controls you list)
     - grass_pct_mean, shrub_pct_mean, tree_pct_mean
     - area_km2
     - pct_high_severity, pct_moderate_severity, etc.  (used to compute severity bins)

Notes:
- Estimation uses pyfixest with two-way fixed effects:
    y ~ i(rel_month, treated, ref=-2) + controls | siteid + time_period
  and HC1 standard errors.
- The script is set up for “wildfire” and “prescribed” (Rx) families.
"""

from __future__ import annotations

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

# =============================================================================
# USER SETTINGS (EDIT THESE)
# =============================================================================

# Where your input CSVs live and where outputs should be written
PROJECT_DIR = Path("path/to/project")  # <-- change
OUTDIR = PROJECT_DIR / "did_outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)

PRED_CSV = PROJECT_DIR / "treatment_control_predictions_with_firedate_relmonth_with_severity.csv"
FIRE_MATRIX_CSV = PROJECT_DIR / "fire_feature_matrix_enhanced.csv"

# Output file name prefix (helps if you run multiple regions/states)
RUN_NAME = "fire_did_12mo"

# Core model settings
PRE_PERIODS_MONTHS = 12
REF_PERIOD_MONTHLY = -2
WEATHER_CONTROLS = ["temperature_mean"]  # add more if you have them, e.g. ["temperature_mean","precip_total"]

# Outcome column in predictions CSV
OUTCOME_COL = "y_pred"

# Optional: exclude site IDs (leave empty list to keep all)
EXCLUDE_SITES: list[str] = []

# =============================================================================
# SMALL UTILITIES
# =============================================================================

def _clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    return df

def _require_cols(df: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

def fdr_bh(pvals, alpha=0.05) -> np.ndarray:
    """Benjamini–Hochberg FDR control. Returns boolean array of rejections."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    if m == 0:
        return np.array([], dtype=bool)

    order = np.argsort(pvals)
    ranked = pvals[order]
    thresh = (np.arange(1, m + 1) / m) * alpha
    passed = ranked <= thresh

    reject = np.zeros(m, dtype=bool)
    if np.any(passed):
        kmax = np.max(np.where(passed)[0])
        reject[: kmax + 1] = True

    out = np.zeros(m, dtype=bool)
    out[order] = reject
    return out

def analyze_pretrends(event_coefs: dict[int, dict], spec_name: str, min_pre_period: int = -12):
    """
    Pre-trend diagnostics for a single spec using pre-period event-study coefficients.
    """
    if not event_coefs:
        return None

    pre = {k: v for k, v in event_coefs.items() if (k < 0 and k >= min_pre_period)}
    if len(pre) < 3:
        return None

    periods = np.array(sorted(pre.keys()), dtype=int)
    b = np.array([pre[k]["coef"] for k in periods], dtype=float)
    se = np.array([pre[k]["se"] for k in periods], dtype=float)

    se_safe = np.where(se <= 0, np.nan, se)
    t = b / se_safe
    p = 2 * (1 - stats.norm.cdf(np.abs(t)))

    n_tests = len(periods)
    bonf_thr = 0.05 / n_tests
    fail_05 = p < 0.05
    fail_bonf = p < bonf_thr
    fail_fdr = fdr_bh(p, alpha=0.05)

    if n_tests >= 4:
        slope_res = stats.linregress(periods, b)
        trend_slope = float(slope_res.slope)
        trend_p = float(slope_res.pvalue)
    else:
        trend_slope = np.nan
        trend_p = np.nan

    return {
        "spec": spec_name,
        "n_pre_periods": int(n_tests),
        "bonf_threshold": float(bonf_thr),
        "n_fail_p05": int(np.sum(fail_05)),
        "n_fail_bonf": int(np.sum(fail_bonf)),
        "n_fail_fdr": int(np.sum(fail_fdr)),
        "mean_pre_coef": float(np.nanmean(b)),
        "sd_pre_coef": float(np.nanstd(b)),
        "max_abs_pre_coef": float(np.nanmax(np.abs(b))),
        "trend_slope": trend_slope,
        "trend_p": trend_p,
        "passes_strict": bool((np.sum(fail_bonf) == 0) and not (trend_p < 0.05)),
        "passes_soft": bool((np.sum(fail_bonf) == 0) and (np.sum(fail_fdr) == 0)),
    }

def aggregate_yearly_FIXED(monthly_coefs: dict[int, dict]):
    """
    Aggregate monthly coefficients into years using inverse-variance weights.
    Year 1 = months 0..11, Year 2 = 12..23, ...; Year 5 = 48+ (collapsed).
    """
    if not monthly_coefs:
        return None

    dfc = pd.DataFrame.from_dict(monthly_coefs, orient="index")
    post = dfc[dfc.index >= 0].copy()
    if post.empty:
        return None

    post["rel_year"] = (np.floor(post.index / 12).astype(int) + 1)
    post.loc[post["rel_year"] > 5, "rel_year"] = 5

    yearly_rows = []
    for y, grp in post.groupby("rel_year"):
        inv_var = 1 / (grp["se"] ** 2 + 1e-10)
        w = inv_var / inv_var.sum()
        coef = float((grp["coef"] * w).sum())
        se = float(1 / np.sqrt(inv_var.sum()))
        yearly_rows.append(
            {
                "rel_year": int(y),
                "coef": coef,
                "se": se,
                "ci_lower": coef - 1.96 * se,
                "ci_upper": coef + 1.96 * se,
                "n_months": int(len(grp)),
                "month_range": f"{int(grp.index.min())}-{int(grp.index.max())}",
            }
        )

    # Reference year 0
    yearly_rows.insert(
        0,
        {
            "rel_year": 0,
            "coef": 0.0,
            "se": 0.0,
            "ci_lower": 0.0,
            "ci_upper": 0.0,
            "n_months": 0,
            "month_range": "ref",
        },
    )
    return {row["rel_year"]: row for row in yearly_rows}

def convert_coefs_to_percent(coefs: dict[int, dict]) -> dict[int, dict]:
    """
    Convert log-point-ish coefficients to percent change: 100*(exp(b)-1).
    If your outcome is already in percent units, replace this with identity.
    """
    out = {}
    for k, v in coefs.items():
        b = float(v["coef"])
        se = float(v["se"])
        # Delta method for SE after exp transform:
        # Var(exp(b)-1) ≈ (exp(b))^2 * Var(b)
        pct = 100.0 * (np.exp(b) - 1.0)
        pct_se = 100.0 * (np.exp(b) * se)
        out[k] = {
            "coef": pct,
            "se": pct_se,
            "ci_lower": pct - 1.96 * pct_se,
            "ci_upper": pct + 1.96 * pct_se,
        }
    return out

def get_ylim_from_coefs(series_map: dict[str, dict[int, dict]], pad=0.10):
    """
    Compute a shared y-limit from multiple percent-coef series.
    series_map maps name -> {time -> {coef,ci_lower,ci_upper}}
    """
    if not series_map:
        return (None, None)

    lows, highs = [], []
    for _, coefs in series_map.items():
        for _, v in coefs.items():
            lows.append(v["ci_lower"])
            highs.append(v["ci_upper"])

    if not lows:
        return (None, None)

    lo = float(np.nanmin(lows))
    hi = float(np.nanmax(highs))
    span = hi - lo
    if span <= 0:
        span = max(abs(lo), abs(hi), 1.0)
    return (lo - pad * span, hi + pad * span)

# =============================================================================
# CLASSIFIERS (FAMILY / VEG / SEVERITY / SIZE)
# =============================================================================

def classify_family(treatment_type) -> str:
    """
    Expected treatment_type strings contain wildfire/rx and treated/control.
    Returns one of:
      wildfire_treated, wildfire_control, rx_treated, rx_control, unknown
    """
    if pd.isna(treatment_type):
        return "unknown"
    tt = str(treatment_type).lower()
    if "wildfire" in tt and "treated" in tt:
        return "wildfire_treated"
    if "wildfire" in tt and "control" in tt:
        return "wildfire_control"
    if ("rx" in tt or "prescribed" in tt) and "treated" in tt:
        return "rx_treated"
    if ("rx" in tt or "prescribed" in tt) and "control" in tt:
        return "rx_control"
    return "unknown"

def classify_vegetation(row) -> str:
    grass = row.get("grass_pct_mean", 0) or 0
    shrub = row.get("shrub_pct_mean", 0) or 0
    tree = row.get("tree_pct_mean", 0) or 0
    if grass == 0 and shrub == 0 and tree == 0:
        return "unknown"
    d = {"grass": grass, "shrub": shrub, "tree": tree}
    return max(d, key=d.get)

def classify_severity(row, veg_type=None) -> str:
    # If you treat grass severity as “low” by construction:
    if veg_type == "grass":
        return "low"
    pct_high = row.get("pct_high_severity", 0) or 0
    pct_mod = row.get("pct_moderate_severity", 0) or 0
    if pct_high > 0.33:
        return "high"
    if (pct_mod + pct_high) > 0.33:
        return "moderate"
    return "low"

def calculate_size_breaks_terciles(df: pd.DataFrame, size_col="area_km2"):
    treated = df[df["treated"] == 1].copy()
    sizes = treated.loc[treated[size_col].notna(), size_col].values
    if len(sizes) < 3:
        return None
    return np.percentile(sizes, [33.33, 66.67])

def classify_fire_size(x, breaks) -> str:
    if pd.isna(x):
        return "unknown"
    if x <= breaks[0]:
        return "small"
    if x <= breaks[1]:
        return "medium"
    return "large"

# =============================================================================
# PANEL METRICS
# =============================================================================

def panel_site_metrics(panel: pd.DataFrame, label: str):
    out = {
        "spec": label,
        "n_obs": int(len(panel)),
        "n_sites_total": int(panel["siteid"].nunique()),
        "n_sites_treated": int(panel.loc[panel["treated"] == 1, "siteid"].nunique()),
        "n_sites_control": int(panel.loc[panel["treated"] == 0, "siteid"].nunique()),
        "n_obs_treated": int((panel["treated"] == 1).sum()),
        "n_obs_control": int((panel["treated"] == 0).sum()),
        "n_time_periods": int(panel["time_period"].nunique()) if "time_period" in panel.columns else None,
    }
    if "family" in panel.columns:
        fam = panel.groupby("family")["siteid"].nunique().to_dict()
        out["sites_by_family"] = ";".join([f"{k}={int(v)}" for k, v in fam.items()])
    if "veg_type" in panel.columns:
        veg = panel.groupby("veg_type")["siteid"].nunique().to_dict()
        out["sites_by_veg"] = ";".join([f"{k}={int(v)}" for k, v in veg.items()])
    return out

# =============================================================================
# ESTIMATION (MONTHLY EVENT STUDY)
# =============================================================================

def estimate_did_monthly(
    df: pd.DataFrame,
    fire_type: str,
    severity: str | None = None,
    vegetation: str | None = None,
    size_class: str | None = None,
    outcome: str = OUTCOME_COL,
):
    """
    Estimate monthly DiD effects (TWFE) and return:
      event_coefs, n_treated_sites, site_ids_used, panel_metrics
    """
    try:
        import pyfixest as pf
    except ImportError as e:
        raise ImportError("pyfixest is required. Install with: pip install pyfixest") from e

    # Fire-type panel filter
    if fire_type == "wildfire":
        panel = df[df["family"].isin(["wildfire_treated", "wildfire_control"])].copy()
    else:
        panel = df[df["family"].isin(["rx_treated", "rx_control"])].copy()

    # Stratifications
    if size_class:
        treated = panel[(panel["treated"] == 1) & (panel["size_class"] == size_class)]
        control = panel[panel["treated"] == 0]
        if vegetation:
            treated = treated[treated["veg_type"] == vegetation]
            control = control[control["veg_type"] == vegetation]
        panel = pd.concat([treated, control], ignore_index=True)

    elif severity:
        if vegetation:
            panel = panel[panel["veg_type"] == vegetation].copy()
        else:
            # common choice: severity only on tree sites
            panel = panel[panel["veg_type"] == "tree"].copy()
        treated = panel[(panel["treated"] == 1) & (panel["severity_class"] == severity)]
        control = panel[panel["treated"] == 0]
        panel = pd.concat([treated, control], ignore_index=True)

    elif vegetation:
        panel = panel[panel["veg_type"] == vegetation].copy()

    # Clean rel_month window
    panel = panel[panel["rel_month"].notna()].copy()
    panel = panel[(panel["rel_month"] >= -PRE_PERIODS_MONTHS)].copy()
    panel["rel_month"] = panel["rel_month"].astype(int)

    # Sample-size checks
    n_treated = panel.loc[panel["treated"] == 1, "siteid"].nunique()
    n_control = panel.loc[panel["treated"] == 0, "siteid"].nunique()

    label = fire_type
    if size_class:
        label += f"|size={size_class}"
    if severity:
        label += f"|sev={severity}"
    if vegetation:
        label += f"|veg={vegetation}"

    metrics = panel_site_metrics(panel, label=label)
    site_ids_used = sorted(panel["siteid"].dropna().unique().tolist())

    if n_treated < 2 or n_control < 1:
        return None, None, site_ids_used, metrics

    # Controls (only keep what exists)
    controls = [c for c in WEATHER_CONTROLS if c in panel.columns]
    control_str = " + ".join(controls) if controls else "1"

    # TWFE event-study
    formula = f"{outcome} ~ i(rel_month, treated, ref={REF_PERIOD_MONTHLY}) + {control_str} | siteid + time_period"

    model = pf.feols(formula, data=panel, vcov="HC1")
    coef_dict = model.coef().to_dict()
    se_dict = model.se().to_dict()

    event_coefs = {}
    for name in coef_dict.keys():
        m = re.search(r"\[(-?\d+)\]:treated", str(name))
        if m:
            t = int(m.group(1))
            b = float(coef_dict[name])
            se = float(se_dict[name])
            event_coefs[t] = {
                "coef": b,
                "se": se,
                "ci_lower": b - 1.96 * se,
                "ci_upper": b + 1.96 * se,
            }

    # Ensure reference point exists for plotting
    if REF_PERIOD_MONTHLY not in event_coefs:
        event_coefs[REF_PERIOD_MONTHLY] = {"coef": 0.0, "se": 0.0, "ci_lower": 0.0, "ci_upper": 0.0}

    return dict(sorted(event_coefs.items())), int(n_treated), site_ids_used, metrics

# =============================================================================
# LOAD + MERGE
# =============================================================================

print("=" * 80)
print("DiD EVENT-STUDY (12-month pre-window) + SITEID/METRICS LOGGING")
print("=" * 80)

print("\nLoading inputs...")
pred = _clean_cols(pd.read_csv(PRED_CSV))
fire = _clean_cols(pd.read_csv(FIRE_MATRIX_CSV))

_require_cols(pred, ["siteid", "year", "month", "rel_month", "treatment_type", OUTCOME_COL], "predictions CSV")
_require_cols(fire, ["siteid", "year", "month"], "fire matrix CSV")

merge_cols = ["siteid", "year", "month"]

# Keep only needed columns from fire matrix (plus optional vegetation/size/severity inputs)
veg_cols = ["grass_pct_mean", "shrub_pct_mean", "tree_pct_mean"]
size_col = "area_km2"
sev_inputs = ["pct_high_severity", "pct_moderate_severity"]

fire_keep = merge_cols + [c for c in WEATHER_CONTROLS + veg_cols + sev_inputs + [size_col] if c in fire.columns]
fire_subset = fire[fire_keep].copy()

df = pred.merge(fire_subset, on=merge_cols, how="left")

# Optional exclusions
if EXCLUDE_SITES:
    before = df["siteid"].nunique()
    df = df[~df["siteid"].isin(EXCLUDE_SITES)].copy()
    after = df["siteid"].nunique()
    print(f"\nExcluded {before - after} site(s). Remaining sites: {after}")

# Fill missing weather controls (simple median imputation)
for col in WEATHER_CONTROLS:
    if col in df.columns and df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Family / treated
df["family"] = df["treatment_type"].apply(classify_family)
df["treated"] = df["family"].str.contains("treated", na=False).astype(int)

# Time-period fixed effect key
df["time_period"] = df["year"].astype(str) + "_" + df["month"].astype(int).astype(str).str.zfill(2)

# Vegetation class at site level (if available)
if all(c in df.columns for c in veg_cols):
    veg_by_site = df.groupby("siteid")[veg_cols].first().reset_index()
    veg_by_site["veg_type"] = veg_by_site.apply(classify_vegetation, axis=1)
    df = df.merge(veg_by_site[["siteid", "veg_type"]], on="siteid", how="left")
else:
    df["veg_type"] = "unknown"

# Severity class (computed)
df["severity_class"] = df.apply(lambda r: classify_severity(r, veg_type=r.get("veg_type")), axis=1)

# Size classes (terciles on treated sites), if size_col exists
if size_col in df.columns:
    breaks = calculate_size_breaks_terciles(df, size_col=size_col)
    if breaks is None:
        df["size_class"] = "unknown"
    else:
        size_by_site = df.groupby("siteid")[size_col].first().reset_index()
        size_by_site["size_class"] = size_by_site[size_col].apply(lambda x: classify_fire_size(x, breaks))
        df = df.merge(size_by_site[["siteid", "size_class"]], on="siteid", how="left")
        df.loc[df["treated"] == 0, "size_class"] = "control"
else:
    df["size_class"] = "unknown"

print(f"\nMerged panel: {len(df):,} rows | {df['siteid'].nunique()} sites")

# =============================================================================
# RUN ALL SPECS + LOGGING
# =============================================================================

monthly_results = {}
annual_results = {}
pretrend_rows = []

siteid_log: dict[str, list[str]] = {}
metrics_log: list[dict] = []

def log_spec(spec_key: str, site_ids: list[str] | None, met: dict | None):
    if site_ids is not None:
        siteid_log[spec_key] = site_ids
    if met is not None:
        metrics_log.append(met)

print("\nEstimating specifications...")

# 1) Overall
monthly_results["overall"] = {}
annual_results["overall"] = {}
for ft in ["wildfire", "prescribed"]:
    coefs, n_sites, site_ids, met = estimate_did_monthly(df, ft)
    log_spec(f"overall__{ft}", site_ids, met)
    if coefs:
        monthly_results["overall"][ft] = {"coefs": coefs, "n_sites": n_sites}
        annual_results["overall"][ft] = {"coefs": aggregate_yearly_FIXED(coefs), "n_sites": n_sites}
        pt = analyze_pretrends(coefs, f"Overall - {ft}")
        if pt:
            pretrend_rows.append(pt)

# 2) Size (if available)
monthly_results["size"] = {}
annual_results["size"] = {}
if "size_class" in df.columns and (df["size_class"] != "unknown").any():
    for ft in ["wildfire", "prescribed"]:
        monthly_results["size"][ft] = {}
        annual_results["size"][ft] = {}
        for sc in ["small", "medium", "large"]:
            coefs, n_sites, site_ids, met = estimate_did_monthly(df, ft, size_class=sc)
            log_spec(f"size__{ft}__{sc}", site_ids, met)
            if coefs:
                monthly_results["size"][ft][sc] = {"coefs": coefs, "n_sites": n_sites}
                annual_results["size"][ft][sc] = {"coefs": aggregate_yearly_FIXED(coefs), "n_sites": n_sites}
                pt = analyze_pretrends(coefs, f"{ft} - size={sc}")
                if pt:
                    pretrend_rows.append(pt)

# 3) Severity (tree-only default inside estimator)
monthly_results["severity"] = {}
annual_results["severity"] = {}
for ft in ["wildfire", "prescribed"]:
    monthly_results["severity"][ft] = {}
    annual_results["severity"][ft] = {}
    for sev in ["low", "moderate", "high"]:
        coefs, n_sites, site_ids, met = estimate_did_monthly(df, ft, severity=sev)
        log_spec(f"severity_tree__{ft}__{sev}", site_ids, met)
        if coefs:
            monthly_results["severity"][ft][sev] = {"coefs": coefs, "n_sites": n_sites}
            annual_results["severity"][ft][sev] = {"coefs": aggregate_yearly_FIXED(coefs), "n_sites": n_sites}
            pt = analyze_pretrends(coefs, f"{ft} - severity={sev} (tree)")
            if pt:
                pretrend_rows.append(pt)

# 4) Vegetation (tree-by-severity interaction + shrub + grass)
monthly_results["interactions"] = {}
annual_results["interactions"] = {}
for ft in ["wildfire", "prescribed"]:
    monthly_results["interactions"][ft] = {}
    annual_results["interactions"][ft] = {}

    # Tree-by-severity (explicit)
    for sev in ["low", "moderate", "high"]:
        coefs, n_sites, site_ids, met = estimate_did_monthly(df, ft, severity=sev, vegetation="tree")
        log_spec(f"int_tree__{ft}__{sev}", site_ids, met)
        if coefs:
            key = f"{sev}_tree"
            monthly_results["interactions"][ft][key] = {"coefs": coefs, "n_sites": n_sites}
            annual_results["interactions"][ft][key] = {"coefs": aggregate_yearly_FIXED(coefs), "n_sites": n_sites}
            pt = analyze_pretrends(coefs, f"{ft} - {sev} x tree")
            if pt:
                pretrend_rows.append(pt)

    # Shrub (no severity split)
    coefs, n_sites, site_ids, met = estimate_did_monthly(df, ft, vegetation="shrub")
    log_spec(f"veg__{ft}__shrub", site_ids, met)
    if coefs:
        monthly_results["interactions"][ft]["shrub"] = {"coefs": coefs, "n_sites": n_sites}
        annual_results["interactions"][ft]["shrub"] = {"coefs": aggregate_yearly_FIXED(coefs), "n_sites": n_sites}
        pt = analyze_pretrends(coefs, f"{ft} - shrub")
        if pt:
            pretrend_rows.append(pt)

    # Grass
    coefs, n_sites, site_ids, met = estimate_did_monthly(df, ft, vegetation="grass")
    log_spec(f"veg__{ft}__grass", site_ids, met)
    if coefs:
        monthly_results["interactions"][ft]["grass"] = {"coefs": coefs, "n_sites": n_sites}
        annual_results["interactions"][ft]["grass"] = {"coefs": aggregate_yearly_FIXED(coefs), "n_sites": n_sites}
        pt = analyze_pretrends(coefs, f"{ft} - grass")
        if pt:
            pretrend_rows.append(pt)

# =============================================================================
# SAVE TABLE OUTPUTS
# =============================================================================

if pretrend_rows:
    pretrend_df = pd.DataFrame(pretrend_rows)
    pretrend_path = OUTDIR / f"{RUN_NAME}_pretrend_summary.csv"
    pretrend_df.to_csv(pretrend_path, index=False)
    print(f"\nSaved pre-trend summary: {pretrend_path.name}")

# Site IDs by spec (long format)
rows = []
for spec, ids in siteid_log.items():
    for sid in (ids or []):
        rows.append({"spec": spec, "siteid": sid})

if rows:
    siteids_df = pd.DataFrame(rows)
    siteids_path = OUTDIR / f"{RUN_NAME}_siteids_by_spec.csv"
    siteids_df.to_csv(siteids_path, index=False)
    print(f"Saved site IDs: {siteids_path.name} "
          f"(specs={siteids_df['spec'].nunique()}, unique_siteids={siteids_df['siteid'].nunique()})")

# Panel metrics by spec
if metrics_log:
    metrics_df = pd.DataFrame(metrics_log)
    metrics_path = OUTDIR / f"{RUN_NAME}_panel_metrics_by_spec.csv"
    metrics_df.to_csv(metrics_path, index=False)
    print(f"Saved panel metrics: {metrics_path.name}")